In [2]:
file_path = 'test_data.txt'

with open(file_path, 'r') as f:
    data = f.read()

In [3]:
print(data)

# Examples:

get_uniprot('P11473')
>>> <Response [200]>

get_uniprot('helloworld')
>>> <Response [400]>

get_uniprot('helloworld').json()
>>> {'url': 'http://rest.uniprot.org/uniprotkb/accessions',
 'messages': ["Accession 'helloworld' has invalid format. It should be a valid UniProtKB accession with optional sequence range e.g. P12345[10-20]."]}

uniprot_parse_response(get_uniprot('P11473'))
>>>
{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
       'source': 'HGNC',
       'id': 'HGNC:12679'}],
     'value': 'VDR'},
    'synonyms': [{'value': 'NR1I1'}]}],
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVS

In [23]:
ensembl_parse_response(get_ensembl('ENSMUSG00000041147'))

{'ENSMUSG00000041147': {'object_type': 'Gene',
  'assembly_name': 'GRCm39',
  'species': 'mus_musculus',
  'db_type': 'core',
  'biotype': 'protein_coding',
  'display_name': 'Brca2',
  'id': 'ENSMUSG00000041147',
  'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
  'canonical_transcript': 'ENSMUST00000044620.11',
  'source': 'ensembl_havana'}}

In [24]:
import requests
import json
import re
import pandas as pd
import numpy as np


def get_uniprot(accession):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    r = requests.get(url)
    return r


def uniprot_parse_response(resp):
    data = resp.json()

    if not resp.ok:
        messages = data.get("messages", ["unknown error"])
        return {"messages": messages}

    accession = data.get("primaryAccession")
    return {
        accession: {
            "organism": data.get("organism", {}).get("scientificName"),
            "geneInfo": data.get("genes", []),
            "sequenceInfo": data.get("sequence", {}),
            "type": "protein"
        }
    }


def get_ensembl(gene_id):
    server = "https://rest.ensembl.org"
    ext = f"/lookup/id/{gene_id}"
    r = requests.get(server + ext, headers={"Content-Type": "application/json"})
    return r


def ensembl_parse_response(resp):
    data = resp.json()

    if not resp.ok:
        messages = data.get("messages", ["unknown error"])
        return {"messages": messages}

    fields = ['object_type', 'assembly_name', 'species', 'db_type', 'biotype',
              'display_name', 'id', 'description', 'canonical_transcript', 'source']
    parsed = {field: data.get(field) for field in fields}
    return {data['id']: parsed}


def main(ids: list):
    result = {}

    for id in ids:
        if id.startswith("ENS"):
            parsed = ensembl_parse_response(get_ensembl(id))
        else:
            parsed = uniprot_parse_response(get_uniprot(id))

        if "messages" in parsed:
            parsed = {id: {"error": "unknown database"}}

        result.update(parsed)

    return pd.DataFrame.from_dict(result, orient='index')


In [25]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])

,organism,geneInfo,sequenceInfo,type,error,object_type,assembly_name,species,db_type,biotype,display_name,id,description,canonical_transcript,source
P11473,Homo sapiens,[{'geneName': {'evidences': [{'evidenceCode': ...,{'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFH...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q91XI3,Ictidomys tridecemlineatus,[{'geneName': {'value': 'INS'}}],{'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHL...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hello,NaN,NaN,NaN,NaN,unknown database,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENSG00000157764,NaN,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRAF,ENSG00000157764,"B-Raf proto-oncogene, serine/threonine kinase ...",ENST00000646891.2,ensembl_havana
ENSG00000139618,NaN,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRCA2,ENSG00000139618,BRCA2 DNA repair associated [Source:HGNC Symbo...,ENST00000380152.8,ensembl_havana
